In [1]:
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
import random

In [12]:
# Some useful constants

nouns = ["Cabbages", "Unicorns", "Toasters", "Penguins", "Bananas", "Zombies", "Rainbows", "Eels", "Pickles", "Muffins"]
adjectives = ["outrageous", "smelly", "pedantic", "existential", "moody", "sparkly", "untrustworthy", "sarcastic", "squishy", "haunted"]

In [2]:

load_dotenv(override=True)

True

In [3]:
def shout(text: Annotated[str, "something to be shouted"]) -> str:
    print(text.upper())
    return text.upper()

shout("hello")

HELLO


'HELLO'

In [4]:
class State(BaseModel):
        
    messages: Annotated[list, add_messages]

In [5]:
graph_builder = StateGraph(State)

In [6]:
def our_first_node(old_state: State) -> State:

    reply = f"{random.choice(nouns)} are {random.choice(adjectives)}"
    messages = [{"role": "assistant", "content": reply}]

    new_state = State(messages=messages)

    return new_state

graph_builder.add_node("first_node", our_first_node)

In [7]:
graph_builder.add_edge(START, "first_node")
graph_builder.add_edge("first_node", END)

In [8]:
graph = graph_builder.compile()

In [10]:
display(Image(graph.get_graph().draw_mermaid_png(max_retries=5, retry_delay=2.0)))

KeyboardInterrupt: 

In [13]:
def chat(user_input: str, history):
    message = {"role": "user", "content": user_input}
    messages = [message]
    state = State(messages=messages)
    result = graph.invoke(state)
    print(result)
    return result["messages"][-1].content


gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


{'messages': [HumanMessage(content='g', additional_kwargs={}, response_metadata={}, id='a1bf29f6-6361-4f47-89e3-b92a60a2bfb5'), AIMessage(content='Cabbages are pedantic', additional_kwargs={}, response_metadata={}, id='cbed64cf-34db-4fad-930e-96f35184fc3d')]}
{'messages': [HumanMessage(content='sd', additional_kwargs={}, response_metadata={}, id='16c750e2-4791-4a1a-832d-e738b11f0881'), AIMessage(content='Bananas are sparkly', additional_kwargs={}, response_metadata={}, id='6eef751f-b999-4834-8a92-9de2859f76de')]}
{'messages': [HumanMessage(content='adsf', additional_kwargs={}, response_metadata={}, id='70cddca2-843f-43fb-bb7b-dbd9b7da2157'), AIMessage(content='Cabbages are untrustworthy', additional_kwargs={}, response_metadata={}, id='916ce547-6f89-455d-9ea1-9d968896f0d6')]}
{'messages': [HumanMessage(content='asdf', additional_kwargs={}, response_metadata={}, id='13fbc88d-d801-4589-8855-3c936d13c652'), AIMessage(content='Unicorns are sarcastic', additional_kwargs={}, response_metadat

In [14]:
llm = ChatOpenAI(model="gpt-4o-mini")

def chatbot_node(old_state: State) -> State:
    response = llm.invoke(old_state.messages)
    new_state = State(messages=[response])
    return new_state

graph_builder.add_node("chatbot", chatbot_node)

Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.


In [16]:
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.


In [ ]:
graph = graph_builder.compile()

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`